<a href="https://colab.research.google.com/github/JavierVazquezPech/TAREAS-IA/blob/main/Ejercicio_8/Ejercicio_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
from dataclasses import dataclass, field


In [2]:
# ==============================================================================
# 1. DEFINICIÓN DEL ENTORNO (qlearn.env)
# ==============================================================================

@dataclass
class Transition:
    reward: float
    nxt: str
    done: bool

class Env:
    def __init__(self):
        # Dimensiones de la cuadrícula: 4 filas (0 a 3), 12 columnas (0 a 11)
        self.rows = 4
        self.cols = 12

        # Acciones en orden: Norte, Este, Sur, Oeste
        self.actions = ("N", "E", "S", "W")

        # Estado Inicial (S) y Meta (G)
        self.start = "3,0"
        self.goal = "3,11"

        # El acantilado (Cliff) va desde la columna 1 hasta la 10 en la fila 3
        self.cliff = {f"3,{c}" for c in range(1, 11)}

        # Estados no terminales (todos excepto la meta G)
        self.nonterminal = []
        for r in range(self.rows):
            for c in range(self.cols):
                state = f"{r},{c}"
                if state != self.goal:
                    self.nonterminal.append(state)

        # Hiperparámetros de entrenamiento por defecto
        self.gamma = 1.0
        self.alpha = 0.5
        self.epsilon = 0.1
        self.episodes = 500
        self.max_steps = 100

    def step(self, state: str, action: str) -> Transition:
        r, c = map(int, state.split(","))

        # Calcular nueva posición tentativa
        if action == "N":
            r = max(0, r - 1)
        elif action == "E":
            c = min(self.cols - 1, c + 1)
        elif action == "S":
            r = min(self.rows - 1, r + 1)
        elif action == "W":
            c = max(0, c - 1)

        nxt = f"{r},{c}"

        # Verificar si cayó en el acantilado
        if nxt in self.cliff:
            return Transition(reward=-100.0, nxt=self.start, done=False)

        # Verificar si llegó a la meta
        if nxt == self.goal:
            return Transition(reward=-1.0, nxt=nxt, done=True)

        # Movimiento normal dentro de la cuadrícula
        return Transition(reward=-1.0, nxt=nxt, done=False)


In [3]:
# ==============================================================================
# 2. FUNCIONES PROPORCIONADAS
# ==============================================================================

def empty_q(env: Env) -> dict[str, dict[str, float]]:
    return {s: {a: 0.0 for a in env.actions} for s in env.nonterminal}

def max_q(Q: dict[str, dict[str, float]], state: str, actions: tuple[str, ...]) -> float:
    if state not in Q:
        return 0.0
    return max(Q[state][a] for a in actions)

def argmax_action(Q: dict[str, dict[str, float]], state: str, actions: tuple[str, ...]) -> str:
    best = actions[0]
    best_v = Q[state][best]
    for a in actions[1:]:
        if Q[state][a] > best_v:
            best, best_v = a, Q[state][a]
    return best

def epsilon_greedy(
    Q: dict[str, dict[str, float]],
    state: str,
    actions: tuple[str, ...],
    epsilon: float,
    rng: random.Random,
) -> str:
    if state not in Q:
        return actions[0]
    if rng.random() < epsilon:
        return rng.choice(list(actions))
    return argmax_action(Q, state, actions)

def td_target_q(env: Env, Q: dict[str, dict[str, float]], reward: float, nxt: str, done: bool) -> float:
    future = 0.0 if done else max_q(Q, nxt, env.actions)
    return reward + env.gamma * future

def td_target_sarsa(
    env: Env, Q: dict[str, dict[str, float]], reward: float, nxt: str, next_action: str | None, done: bool
) -> float:
    if done or next_action is None or nxt not in Q:
        future = 0.0
    else:
        future = Q[nxt][next_action]
    return reward + env.gamma * future

def apply_update(Q: dict[str, dict[str, float]], state: str, action: str, target: float, alpha: float) -> float:
    old = Q[state][action]
    Q[state][action] = old + alpha * (target - old)
    return Q[state][action]

def train(
    env: Env, rng: random.Random, episodes: int | None = None, epsilon: float | None = None, method: str = "q-learning"
) -> tuple[dict[str, dict[str, float]], list[float]]:
    if method not in {"q-learning", "sarsa"}:
        raise ValueError("method must be q-learning or sarsa")
    n = env.episodes if episodes is None else episodes
    eps = env.epsilon if epsilon is None else epsilon
    Q = empty_q(env)
    returns: list[float] = []
    for _ in range(n):
        s = env.start
        G = 0.0
        a = epsilon_greedy(Q, s, env.actions, eps, rng)
        for _step in range(env.max_steps):
            if s not in Q:
                break
            tr = env.step(s, a)
            G += tr.reward
            if method == "q-learning":
                target = td_target_q(env, Q, tr.reward, tr.nxt, tr.done)
                apply_update(Q, s, a, target, env.alpha)
                s = tr.nxt
                if tr.done:
                    break
                a = epsilon_greedy(Q, s, env.actions, eps, rng)
            else:
                a2 = None if tr.done or tr.nxt not in Q else epsilon_greedy(Q, tr.nxt, env.actions, eps, rng)
                target = td_target_sarsa(env, Q, tr.reward, tr.nxt, a2, tr.done)
                apply_update(Q, s, a, target, env.alpha)
                s = tr.nxt
                if tr.done:
                    break
                a = a2 if a2 is not None else env.actions[0]
        returns.append(G)
    return Q, returns

def greedy_policy(env: Env, Q: dict[str, dict[str, float]]) -> dict[str, str]:
    return {s: argmax_action(Q, s, env.actions) for s in env.nonterminal}

def greedy_return(env: Env, Q: dict[str, dict[str, float]]) -> float:
    s = env.start
    G = 0.0
    for _ in range(env.max_steps):
        if s not in Q:
            break
        a = argmax_action(Q, s, env.actions)
        tr = env.step(s, a)
        G += tr.reward
        s = tr.nxt
        if tr.done:
            break
    return G


In [4]:
# ==============================================================================
# 3. EJECUCIÓN Y COMPROBACIONES
# ==============================================================================

def moving_average(data: list[float], window: int = 10) -> list[float]:
    return [sum(data[max(0, i - window + 1):i + 1]) / min(i + 1, window) for i in range(len(data))]

def print_grid_policy(env: Env, policy: dict[str, str]):
    arrows = {"N": "↑", "E": "→", "S": "↓", "W": "←"}
    for r in range(env.rows):
        row_str = []
        for c in range(env.cols):
            state = f"{r},{c}"
            if state == env.goal:
                row_str.append(" G ")
            elif state in env.cliff:
                row_str.append(" C ")
            else:
                row_str.append(f" {arrows[policy[state]]} ")
        print("".join(row_str))

if __name__ == "__main__":
    env = Env()

    # --------------------------------------------------------------------------
    # Prueba Manual de Transiciones
    # --------------------------------------------------------------------------
    print("--- COMPROBACIÓN MANUAL DE CASOS ---")

    # Caso 1: Un paso norte desde S -> (2,0), r = -1
    t1 = env.step(env.start, "N")
    print(f"1. Norte desde S ({env.start}): Cae en {t1.nxt}, r = {t1.reward}, done = {t1.done}")

    # Caso 2: Un paso este desde S -> Cae en S, r = -100, el episodio sigue (done=False)
    t2 = env.step(env.start, "E")
    print(f"2. Este desde S ({env.start}):  Cae en {t2.nxt}, r = {t2.reward}, done = {t2.done}")

    # Caso 3: Paso sur desde (2,11) -> Entra a G (3,11), r = -1, done = True
    t3 = env.step("2,11", "S")
    print(f"3. Sur desde (2,11):       Cae en {t3.nxt}, r = {t3.reward}, done = {t3.done}\n")

    # --------------------------------------------------------------------------
    # Entrenamiento y Resultados
    # --------------------------------------------------------------------------
    SEED = 42

    for algo in ["q-learning", "sarsa"]:
        rng = random.Random(SEED)
        Q_table, train_returns = train(env, rng, method=algo)
        pol = greedy_policy(env, Q_table)
        ret_greedy = greedy_return(env, Q_table)

        print(f"--- RESULTADOS CON {algo.upper()} ---")
        print(f"Retorno de la política Greedy: {ret_greedy}")
        print("Representación de la política óptima descubierta:")
        print_grid_policy(env, pol)

        # Mostrar una pequeña muestra de la Media Móvil (últimos episodios)
        ma = moving_average(train_returns, window=10)
        print(f"Media móvil (últimos 5 episodios de entrenamiento): {[round(x, 1) for x in ma[-5:]]}\n")


--- COMPROBACIÓN MANUAL DE CASOS ---
1. Norte desde S (3,0): Cae en 2,0, r = -1.0, done = False
2. Este desde S (3,0):  Cae en 3,0, r = -100.0, done = False
3. Sur desde (2,11):       Cae en 3,11, r = -1.0, done = True

--- RESULTADOS CON Q-LEARNING ---
Retorno de la política Greedy: -13.0
Representación de la política óptima descubierta:
 ↑  →  →  →  ←  ↑  ↓  →  ↓  →  ↓  ↓ 
 ←  ↓  →  →  →  →  ↓  →  →  ↓  →  ↓ 
 →  →  →  →  →  →  →  →  →  →  →  ↓ 
 ↑  C  C  C  C  C  C  C  C  C  C  G 
Media móvil (últimos 5 episodios de entrenamiento): [-36.3, -47.6, -58.9, -59.1, -59.0]

--- RESULTADOS CON SARSA ---
Retorno de la política Greedy: -17.0
Representación de la política óptima descubierta:
 →  →  →  →  →  →  →  →  →  →  →  ↓ 
 →  ↑  ↑  ↑  →  →  →  ↑  ←  →  →  ↓ 
 ↑  ↑  ↑  ↑  →  ↑  ↑  ←  ↑  ←  →  ↓ 
 ↑  C  C  C  C  C  C  C  C  C  C  G 
Media móvil (últimos 5 episodios de entrenamiento): [-19.4, -19.1, -19.7, -19.4, -19.5]

